# Implementing Knowledge Distillation

This notebook walks through a complete implementation of knowledge distillation with working code examples. We'll build everything from scratch and train a real student model on MNIST.

The teacher is trained first and then frozen.
The distilled student is compared against a control student with the same architecture, initial weights, batch order and number of epochs, trained on the labels alone.
Choices are made on a validation split, and the test set is only read for the final comparison.

## Setup and Imports

In [ ]:
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
SEED = 0
# downloads land outside the repository, set KD_DATA_DIR to put them somewhere else
DATA_DIR = os.environ.get('KD_DATA_DIR', os.path.join(os.path.expanduser('~'), '.cache', 'kd-data'))

## Step 1: Temperature Scaling Function

Temperature controls how "smooth" the probability distribution becomes. Let's implement and test it:

In [ ]:
def apply_temperature(logits, temperature):
    """
    temperature scaled softmax, the soft targets distillation trains on

    args:
        logits: raw model outputs before softmax
        temperature: higher gives a softer distribution, 1 is the ordinary softmax

    returns:
        softened probability distribution
    """
    return F.softmax(logits / temperature, dim=-1)

In [ ]:
# same example logits as notebook 01
logits = torch.tensor([10.0, 2.0, 1.0, 0.5])

print("temp scaling examples:")
print("="*60)
for T in [1, 3, 5, 10]:
    probs = apply_temperature(logits, temperature=T)
    print(f"\nT={T:2d}: {probs.numpy()}")
    print(f"Entropy: {-(probs * torch.log(probs)).sum():.4f} nats")

print("\nHigher temperature = Higher entropy = More of the teacher's ranking of the wrong classes is visible")

## Step 2: The Distillation Loss Function

The core of knowledge distillation - combining soft targets from the teacher with hard labels:

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, temperature=3.0, alpha=0.7):
    """
    knowledge distillation loss after hinton, vinyals and dean (2015)

    args:
        student_logits: raw outputs from the student
        teacher_logits: raw outputs from the frozen teacher
        labels: ground truth class indices
        temperature: softening parameter, above 1 exposes how the teacher ranks wrong classes (default: 3.0)
        alpha: weight on the distillation term, 1 - alpha goes to the label term (default: 0.7)

    returns:
        (total, distill, student) so the two terms can be logged separately
    """
    # kl_div expects the student as log probabilities and the teacher as probabilities
    soft_teacher = F.softmax(teacher_logits / temperature, dim=-1)
    soft_student = F.log_softmax(student_logits / temperature, dim=-1)

    # distillation term: kl(teacher_t || student_t), batchmean gives the per example kl
    # times t^2 because soft target gradients scale as 1 / t^2, this keeps both terms balanced as t changes
    distill_loss = F.kl_div(
        soft_student,
        soft_teacher,
        reduction='batchmean') * (temperature ** 2)

    # label term: ordinary cross entropy at t=1 against the ground truth
    student_loss = F.cross_entropy(student_logits, labels)

    total_loss = alpha * distill_loss + (1 - alpha) * student_loss

    return total_loss, distill_loss, student_loss

In [ ]:
# smoke test on random logits
torch.manual_seed(SEED)
dummy_student = torch.randn(4, 10)
dummy_teacher = torch.randn(4, 10)
dummy_labels = torch.randint(0, 10, (4,))

total, distill, student = distillation_loss(dummy_student, dummy_teacher, dummy_labels, temperature=3.0, alpha=0.7)

print("Testing Distillation Loss Function:")
print(f"Total Loss: {total:.4f}")
print(f"Distillation Loss: {distill:.4f} (weight: 0.7)")
print(f"Student Loss: {student:.4f} (weight: 0.3)")
print(f"\nVerify: {0.7*distill + 0.3*student:.4f} = {total:.4f}")

## Step 3: Define Teacher and Student Models

Two MLPs for MNIST, both returning logits:

In [ ]:
class TeacherModel(nn.Module):
    """large mlp teacher, about 13x the student's parameters"""
    def __init__(self, input_size=784, num_classes=10):
        super(TeacherModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
class StudentModel(nn.Module):
    """small mlp student, the model meant for deployment"""
    def __init__(self, input_size=784, num_classes=10):
        super(StudentModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
def new_model(model_class, seed):
    # the seed fixes the initial weights, so a control and a distilled student can start identical
    torch.manual_seed(seed)
    return model_class().to(device)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

teacher_params = count_parameters(TeacherModel())
student_params = count_parameters(StudentModel())

In [ ]:
print("model architecture comparison:")
print(f"Teacher parameters: {teacher_params:,}")
print(f"Student parameters: {student_params:,}")
print(f"Compression ratio: {teacher_params/student_params:.1f}x")
print(f"Size reduction: {(1-student_params/teacher_params)*100:.1f}%")

In [ ]:
print("Teacher Architecture:")
print(TeacherModel())
print("\nStudent Architecture:")
print(StudentModel())

## Step 4: Load MNIST

The images are flattened to 784 values and standardised with the usual MNIST mean and standard deviation.
6,000 training images are held out as a validation set.

In [ ]:
def load_mnist(train):
    # the whole set fits in memory, so the tensors are built once instead of decoding images per batch
    raw = datasets.MNIST(DATA_DIR, train=train, download=True)
    x = (raw.data.float() / 255.0 - 0.1307) / 0.3081
    return x.view(len(x), -1), raw.targets.clone()

In [ ]:
train_X_full, train_y_full = load_mnist(train=True)
test_X, test_y = load_mnist(train=False)

perm = torch.randperm(len(train_X_full), generator=torch.Generator().manual_seed(SEED))
val_X, val_y = train_X_full[perm[:6000]], train_y_full[perm[:6000]]
train_X, train_y = train_X_full[perm[6000:]], train_y_full[perm[6000:]]

In [ ]:
def make_train_loader(seed, batch_size=128):
    # a generator per run: the same seed gives the control and the distilled student the same batches
    return DataLoader(TensorDataset(train_X, train_y), batch_size=batch_size, shuffle=True,
                      generator=torch.Generator().manual_seed(seed))

val_loader = DataLoader(TensorDataset(val_X, val_y), batch_size=1000)
test_loader = DataLoader(TensorDataset(test_X, test_y), batch_size=1000)

In [ ]:
print(f"Training samples: {len(train_X)}")
print(f"Validation samples: {len(val_X)}")
print(f"Test samples: {len(test_X)}")
print(f"Batches per epoch at batch size 128: {len(make_train_loader(SEED))}")

## Step 5: Evaluation

Accuracy with dropout off, the argmax of the logits is the same at every temperature:

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            predictions = torch.argmax(outputs, dim=1)
            correct += (predictions == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = 100 * correct / total
    return accuracy

## Step 6: Pre-train the Teacher Model

First, we need a well-trained teacher. Let's train it on the labels alone:

In [ ]:
# plain supervised training on hard labels, reused below for the no distillation control
def train_supervised(model, train_loader, num_epochs=10, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr)

    losses = []
    val_accs = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = F.cross_entropy(outputs, batch_y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        losses.append(epoch_loss / len(train_loader))
        val_accs.append(evaluate_model(model, val_loader))

    return losses, val_accs

In [ ]:
num_epochs = 10

In [ ]:
t0 = time.time()
teacher = new_model(TeacherModel, SEED)
teacher_losses, teacher_val_accs = train_supervised(teacher, make_train_loader(SEED), num_epochs=num_epochs)
print(f"teacher trained in {time.time() - t0:.0f} s")
print(f"teacher validation accuracy by epoch: {[round(a, 2) for a in teacher_val_accs]}")

In [ ]:
# freeze the teacher: eval mode turns its dropout off and no gradient will ever reach it
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

## Step 7: Train Student with Distillation

Now train the student model using knowledge distillation:

In [ ]:
def train_student_distillation(student_model, teacher_model, train_loader, num_epochs=10, temperature=4.0, alpha=0.7, lr=0.001):
    """
    train the student on the frozen teacher's softened outputs plus the true labels
    """
    optimizer = optim.Adam(student_model.parameters(), lr=lr)
    teacher_model.eval()  # dropout off, so the soft targets are deterministic

    total_losses = []
    distill_losses = []
    label_losses = []
    val_accs = []

    for epoch in range(num_epochs):
        student_model.train()
        epoch_loss = 0
        epoch_distill = 0
        epoch_label = 0

        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            optimizer.zero_grad()

            # no_grad: the teacher is frozen, only the student's optimizer steps
            with torch.no_grad():
                teacher_logits = teacher_model(batch_X)

            student_logits = student_model(batch_X)

            total_loss, distill_loss, label_loss = distillation_loss(student_logits, teacher_logits, batch_y, temperature=temperature, alpha=alpha)

            total_loss.backward()
            optimizer.step()

            epoch_loss += total_loss.item()
            epoch_distill += distill_loss.item()
            epoch_label += label_loss.item()

        total_losses.append(epoch_loss / len(train_loader))
        distill_losses.append(epoch_distill / len(train_loader))
        label_losses.append(epoch_label / len(train_loader))
        val_accs.append(evaluate_model(student_model, val_loader))

    return total_losses, distill_losses, label_losses, val_accs

In [ ]:
temperature = 4.0
alpha = 0.7

In [ ]:
t0 = time.time()
student = new_model(StudentModel, SEED)
total_losses, distill_losses, label_losses, student_val_accs = train_student_distillation(
    student, teacher, make_train_loader(SEED), num_epochs=num_epochs, temperature=temperature, alpha=alpha)
print(f"distilled student trained in {time.time() - t0:.0f} s")

## Step 8: Train the Control Student

The same architecture, seed, batches and epochs, trained without the teacher:

In [ ]:
t0 = time.time()
baseline_student = new_model(StudentModel, SEED)
baseline_losses, baseline_val_accs = train_supervised(baseline_student, make_train_loader(SEED), num_epochs=num_epochs)
print(f"control student trained in {time.time() - t0:.0f} s")

## Step 9: Visualize Training Progress

Validation accuracy of all three models, and how the distilled student's two loss terms evolved:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
epochs_axis = range(1, num_epochs + 1)

axes[0].plot(epochs_axis, teacher_val_accs, label='Teacher', marker='o', linewidth=2)
axes[0].plot(epochs_axis, student_val_accs, label='Student (distilled)', marker='s', linewidth=2)
axes[0].plot(epochs_axis, baseline_val_accs, label='Student (control)', marker='^', linewidth=2)
axes[0].set_title('Validation Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy (%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_axis, distill_losses, label='Distillation Loss (from teacher, times T^2)',
             marker='^', linewidth=2, color='orange')
axes[1].plot(epochs_axis, label_losses, label='Student Loss (from labels)',
             marker='v', linewidth=2, color='green')
axes[1].plot(epochs_axis, total_losses, label='Total Loss',
             marker='s', linewidth=2, color='blue', linestyle='--')
axes[1].set_title('Distilled Student Loss Components', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 10: Hyperparameter Exploration

Let's experiment with different temperatures and alpha values, scored on the validation set:

In [ ]:
# t and alpha change together in each config, so their effects cannot be separated
configs = [
    {'temperature': 2.0, 'alpha': 0.5},
    {'temperature': 4.0, 'alpha': 0.7},
    {'temperature': 6.0, 'alpha': 0.9},]

results = []
for config in configs:
    test_student = new_model(StudentModel, SEED)
    *_, val_accs = train_student_distillation(
        test_student, teacher, make_train_loader(SEED),
        num_epochs=num_epochs,
        temperature=config['temperature'],
        alpha=config['alpha'])
    results.append({
        'Temperature': config['temperature'],
        'Alpha': config['alpha'],
        'Validation Accuracy': val_accs[-1]})

In [ ]:
df_results = pd.DataFrame(results)
print("Results Summary:")
print(df_results.to_string(index=False))
print("\nBest configuration on validation:", df_results.loc[df_results['Validation Accuracy'].idxmax()].to_dict())

## Step 11: Evaluate and Compare Models

The test set is read here for the first time:

In [ ]:
teacher_acc = evaluate_model(teacher, test_loader)
student_acc = evaluate_model(student, test_loader)
baseline_acc = evaluate_model(baseline_student, test_loader)

In [ ]:
print("Final Comparison (test accuracy):")
print(f"Teacher (large model): {teacher_acc:.2f}%")
print(f"Student with Distillation: {student_acc:.2f}%")
print(f"Student without Distillation: {baseline_acc:.2f}%")
print(f"\nDistilled minus control: {student_acc - baseline_acc:+.2f} percentage points")
print(f"Gap to Teacher (with distillation): {teacher_acc - student_acc:.2f} percentage points")
print(f"Gap to Teacher (control): {teacher_acc - baseline_acc:.2f} percentage points")
print(f"Compression: {teacher_params/student_params:.1f}x smaller")

## Key Takeaways

1. **The loss**: `alpha * T^2 * KL(teacher_T || student_T) + (1 - alpha) * CE(labels, student)`, computed on logits, with the teacher frozen.
2. **The control**: a distillation result only means something next to a student with the same architecture, seed, batches and epochs trained without the teacher.
3. **Validation, then test**: hyperparameters are chosen on held out training data, the test set is read once at the end.